# 🧩 Can you rediscover the 7 hidden personas?
Unsupervised learning on a dataset with a secret answer key (`archetype`). K-Means, GMM, hierarchical, PCA, t-SNE.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, warnings
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
D = pd.read_csv('/kaggle/input/datasets/pranavsinghdeveloper/desi-cs-student-survival/students.csv')
S = pd.read_csv('/kaggle/input/datasets/pranavsinghdeveloper/desi-cs-student-survival/semester_records.csv')

In [ ]:
from sklearn.preprocessing import StandardScaler; from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, AgglomerativeClustering; from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA; from sklearn.metrics import adjusted_rand_score, silhouette_score
cols=['attendance_pct','study_hours_daily','gaming_hours_weekly','sleep_hours','leetcode_solved','communication_skill','club_memberships','linkedin_connections','certifications_count','chai_cups_per_day','bunked_classes_per_month','cgpa']
Z=StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(np.log1p(D[cols].clip(lower=0))))
ks=range(2,12); sil=[silhouette_score(Z,KMeans(k,n_init=5,random_state=0).fit_predict(Z),sample_size=3000,random_state=0) for k in ks]
plt.plot(list(ks),sil,marker='o'); plt.xlabel('k'); plt.ylabel('silhouette'); plt.title('Silhouette vs k (truth is 7)'); plt.show()

## Compare algorithms against the answer key (ARI)

In [ ]:
res={'KMeans':KMeans(7,n_init=10,random_state=0).fit_predict(Z),'GMM':GaussianMixture(7,random_state=0,n_init=3).fit_predict(Z),
     'Ward':AgglomerativeClustering(7).fit_predict(Z[:4000])}
for n,l in res.items(): print(n, round(adjusted_rand_score(D.archetype[:len(l)],l),3))

In [ ]:
P=PCA(2).fit_transform(Z); fig,ax=plt.subplots(1,2,figsize=(13,4.5))
sns.scatterplot(x=P[:,0],y=P[:,1],hue=D.archetype,s=6,ax=ax[0]).set(title='True personas'); sns.scatterplot(x=P[:,0],y=P[:,1],hue=res['GMM'],palette='tab10',s=6,ax=ax[1]).set(title='GMM clusters'); plt.tight_layout(); plt.show()
print(D.groupby('archetype')[['cgpa','gaming_hours_weekly','leetcode_solved','club_memberships']].mean().round(1))

## Your turn
Personas overlap by design (traits have noise). Try UMAP+HDBSCAN, feature weighting, or adding categorical columns. Baseline ARI is on the challenges page.

---
Dataset docs, generator & challenges: https://github.com/Pranav-Singh-Devloper/desi-cs-student-survival-dataset